#### This code returns the histograms MATRICES for every day in order to have the coordinates histograms 

In [ ]:
from matplotlib.image import NonUniformImage
import matplotlib.pyplot as plt

## Functions

In [2]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

## Folders

In [3]:
#mag_folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/2014_months/11"
#mag_folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/2015_months/4"
mag_folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/all_data"

lap_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/all_data"
#lap_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2014/noviembre"
#lap_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2015/abril"

## CODE 2

## MAG data using 1 second

In [5]:
#IMPORT DATA
directory= os.scandir(mag_folder)
list_of_files=get_directory(mag_folder)

newlist2=[]
for item in list_of_files:
    newlist2.append(mag_folder+'/'+str(item))
#print(newlist)

mag_table=[]

for j in np.arange(0,len(newlist2)):
#for j in np.arange(0,1):
    print(j)
    #Importing data
    titles=['Dates_and_Hours', '?', 'x', 'y', 'z', 'Bx', 'By', 'Bz', 'flag']
    data= pd.read_table(str(newlist2[j]), header=None, names=titles, sep='\s+', parse_dates=['Dates_and_Hours'])
    #print(data)
     #Filling the data gaps
    data.Dates_and_Hours = data.Dates_and_Hours.dt.round('1s') #round to one second for simplicity
    if data.shape[0] < 86400: # if the number of datapoints is lower than one day:
        #print('Data gaps detected, padding array....')
        data2 = data.rename(index=(data['Dates_and_Hours']-data.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
        data2 = data2.reindex(range(0,86400)) # now we just fill in the missing values
        newt = pd.date_range(start = data.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
        data2['Dates_and_Hours'] = newt # now fill in the times so there is no NaT
        data = data2 
        del(data2)
    elif data.shape[0] > 86400:
        error('Data file is too long, probably need to debug the code again....')
    #print('Done\n')
    
    
    #LOADING DATA
    #--------------------------------------------
    XX = data[['Dates_and_Hours','x','y','z']]
    
    #Calculates the root for each point
    roots=[]
    for i in np.arange(0,len(XX)):
        rootpoint=(XX['y'][i]**2+XX['z'][i]**2)**(1/2)
        roots.append(rootpoint)
    
    #Transform a list into a data column
    df1 = pd.DataFrame({'col':roots})
    XX['roots']=df1
    
    mag_table.append(XX)

0


<ipython-input-5-d948b1da8109>:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  XX['roots']=df1


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128


C:\Users\Ariel\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3338: DtypeWarning: Columns (8) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
288
289
290
291
292
293
294
295
296
297
298
299
300
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
337
338
339
340
341
342
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
378


In [6]:
#-----------------------------------------

#IMPORT DATA:
directory= os.scandir(lap_folder)
list_of_files=get_directory(lap_folder)

newlist=[]
for item in list_of_files:
    newlist.append(lap_folder+'/'+str(item))

print(newlist)    
H3=[]
H4=[]

#for i in np.arange(1,2): #For one day
for i in np.arange(235,len(newlist)): #For every day
        print(i)
        #Lap data
        #--------------------------------------------
        title2=['t_utc','?','npl','??','???','????']
        path= str(newlist[i])
        data1= pd.read_table(path, header=None, names=title2, sep=',', engine='python', parse_dates=['t_utc'])
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        data2=data2.reset_index(drop=True)
        data2=data2[['t_utc','npl']]
        #Resample
        data2['index'] = pd.to_datetime(data2['t_utc'])
        data2.set_index('index', inplace=True)
        data2=data2.resample('1S').mean()
        data2['t_utc'] = pd.to_datetime(data2.index.values)
        data2=data2.dropna() #Drop the nan values of the resample
        
        path_time= pd.to_datetime(data2['t_utc'])
        q=path_time.dt.year
        qq=path_time.dt.month
        qqq=path_time.dt.day
        qqqq=path_time.dt.hour
        qqqqq=path_time.dt.minute        
        
        #Chosing the specific MAG data
        for j in np.arange(0,len(mag_table)):
            if pd.to_datetime(mag_table[j]['Dates_and_Hours']).dt.year[0]==q[0] and pd.to_datetime(mag_table[j]['Dates_and_Hours']).dt.month[0]==qq[0] and pd.to_datetime(mag_table[j]['Dates_and_Hours']).dt.day[0]==qqq[0]:
                day_chosen=mag_table[j]
                break
        
        day_chosen_copy=day_chosen.copy()
        day_chosen_copy.set_index('Dates_and_Hours', inplace=True)      

        att = day_chosen_copy.reindex(data2['t_utc'], method='pad') 
        
        
        #-----------------------------
        #USED FOR THE ROOTS HISTOGRAM
        final_dataframe=att[['x','roots']]
        import csv
        final_dataframe.to_csv('finaldataframeroots_'+str(i), index=False, sep='\t')    
        #HISTOGRAM FOR ROOT/X
        final_dataframe_copy=final_dataframe.copy()
        #Filter nan values        
        final_dataframe_copy=final_dataframe_copy.dropna(subset=['x','roots'])
        final_dataframe_copy=final_dataframe_copy.reset_index(drop=True)
            
        import math
        x=final_dataframe_copy['x']
        xx=x.astype(float)
        y=final_dataframe_copy['roots']
        yy=y.astype(float) 

        #---------
        #xmin=-160
        #xmax=920
        #ymin=0
        #ymax=1200
        
        xmin=-20
        xmax=200
        ymin=0
        ymax=450
        #---------
        H, xedges, yedges= np.histogram2d(xx,yy, bins=100, range=[[xmin,xmax],[ymin,ymax]])
        H3.append(H)        
        #----------------------------------------
        np.savetxt("matrixcoordinates_"+str(i) +".txt", H)
        
        
        
        
        #-----------------------------
        #USED FOR THE Y VS Z HISTOGRAM
        #final_dataframe=att[['y','z']]
        #import csv
        #final_dataframe.to_csv('finaldataframeyvsz_'+str(i), index=False, sep='\t')    
        #HISTOGRAM FOR Y/Z
        #final_dataframe_copy=final_dataframe.copy()
        #Filter nan values        
        #final_dataframe_copy=final_dataframe_copy.dropna(subset=['y','z'])
        #final_dataframe_copy=final_dataframe_copy.reset_index(drop=True)
            
        #import math
        #x=final_dataframe_copy['z']
        #xx=x.astype(float)
        #y=final_dataframe_copy['y']
        #yy=y.astype(float)  
           
        #---------
        #xmin=-450
        #xmax=450
        #ymin=-1000
        #ymax=400
        #---------
        #H2, xedges, yedges= np.histogram2d(xx,yy, bins=100, range=[[xmin,xmax],[ymin,ymax]])
        #H4.append(H2)
        #np.savetxt("matriz2.txt", H)        
        #----------------------------------------
        #np.savetxt("matrix_yvsz"+str(i) +".txt", H2)
        



['C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141106_130258_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141107_000000_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141109_040314_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141110_000000_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141114_100314_807_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141118_090258_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141120_090258_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141206_011235_805_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141210_090259_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141211_000000_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141212_090259_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/all_data/LAP_20141213_000000_803_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/al

In [7]:
print(H3[0])

extent=[-160,920,0,1200]
plt.imshow(H3[0].T ,origin='lower', extent=extent, aspect='auto')

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


Here I have the histograms matrices for every day 

In [8]:
Done

NameError: name 'Done' is not defined

### Matrix per month

In [ ]:
zeros3 = np.zeros(shape=(100,100))    
for i in np.arange(0,len(H3)):
    zeros3=np.add(zeros3,H3[i])

  
    
print("Addition of two matrix") 
print(zeros3)


plt.figure()

plt.imshow(zeros3.T ,origin='lower', extent=extent, aspect='auto')

#np.savetxt("matrix1.txt", zeros)
#np.savetxt("matrix2.txt", zeros2)
#plt.plot(zeros2)

In [ ]:
print(zeros2)
plt.plot(zeros2)

In [ ]:
DONE

## NOW WE WANT THE X AN ROOT VALUE IN THE FINAL LIST OF MIRROR MODE WAVES

In [ ]:
mag_table22=mag_table.copy()

In [ ]:
MM_prominence_data2="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events3"

#Importing data
titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson','gas_production_rates', 'bmodulus','cometo-centric distance']
data= pd.read_table(str(MM_prominence_data2), header=None, names=titles, sep='\t')
data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
data2=data2.reset_index(drop=True)
print('data2')
print(data2)
path_time= pd.to_datetime(data2['Beginning']) 
path_time2= pd.to_datetime(data2['End']) 
q=path_time.dt.year
qq=path_time.dt.month
qqq=path_time.dt.day


x_list=[]
root_list=[]

for i in np.arange(0,len(data2)):
#for i in np.arange(1,2):
    print(i)
    year=q[i]
    month=qq[i]
    day=qqq[i]
    #print(year)
    #print(month)
    #print(day)
    for j in np.arange(0,len(mag_table22)):
     
        if pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.year[0]==year and pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.month[0]==month and pd.to_datetime(mag_table22[j]['Dates_and_Hours']).dt.day[0]==day:   #
            day_chosen=mag_table22[j]         
            
            # Chosing the specific time interval
            filtered_df = day_chosen.loc[(day_chosen['Dates_and_Hours'] >= path_time[i])  & (day_chosen['Dates_and_Hours'] <= path_time2[i])]
            filtered_df=filtered_df.reset_index(drop=True)
            #print('filtered')
            #print(filtered_df)
            
            # find the mean x and root value of the interval
            listA=filtered_df['x'].tolist()
            mean_val1=np.mean(listA)
            x_list.append(mean_val1)
            
            listB=filtered_df['roots'].tolist()
            mean_val2=np.mean(listB)
            root_list.append(mean_val2)
         
            break

In [ ]:
print(min(x_list))
print(max(x_list))
print(min(root_list))
print(max(root_list))



In [ ]:
data3=data2.copy()
data3['x']=x_list
data3['roots']=root_list
import csv
data3.to_csv('all_prominence_events5', index=False, sep='\t')  
